# Activity-2: Data Storytelling through Visualization
**Bank Marketing Campaign Response & Conversion Analysis** — Interactive dashboard in Python (Plotly + ipywidgets)

Student: Debadrita Saha (25MCAR0028) | MCA-AIML Sem III | Faculty: Prof. Rahul Pawar

Dataset: `bank-additional-full.csv` (Moro, Cortez & Rita, 2014) — 41,188 telemarketing contacts, 21 columns, target `y` = term-deposit subscription.

In [ ]:
# !pip install pandas plotly ipywidgets   # uncomment if needed
import os, pandas as pd, numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. Load data & calculated fields (the Tableau calculated fields, in pandas)

In [ ]:
for p in ['bank-additional-full.csv', '/mnt/user-data/uploads/bank-additional-full.csv']:
    if os.path.exists(p):
        DATA_PATH = p; break
df = pd.read_csv(DATA_PATH, sep=';')           # semicolon-delimited
print(df.shape)

MONTHS = ['mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
df['y_bin'] = (df['y'] == 'yes').astype(int)                                   # Subscribed
df['age_group'] = pd.cut(df['age'], [0,25,35,45,55,65,200],
                         labels=['17-25','26-35','36-45','46-55','56-65','65+'])
df['contact_band'] = pd.cut(df['campaign'], [0,1,2,3,5,1000],
                            labels=['1st contact','2nd contact','3rd contact','4-5 contacts','6+ contacts'])
df['month'] = pd.Categorical(df['month'], categories=MONTHS, ordered=True)
df.head()

## 2. Headline KPIs

In [ ]:
def kpis(d):
    n, s = len(d), d['y_bin'].sum()
    ps = d[d['poutcome'] == 'success']
    return {
        'Total Contacts': f"{n:,}",
        'Subscribed': f"{s:,}",
        'Conversion Rate': f"{100*s/n:.2f}%" if n else '-',
        'Avg Duration (s)': f"{d['duration'].mean():.1f}" if n else '-',
        'Cellular Share': f"{100*(d['contact']=='cellular').mean():.1f}%" if n else '-',
        'Prior Success Conv.': f"{100*ps['y_bin'].mean():.1f}%" if len(ps) else '-',
    }

def kpi_html(d):
    cards = ''.join(f"<div style='flex:1;min-width:130px;border:1px solid #ccd;border-radius:8px;padding:8px 12px;margin:4px'>"
                    f"<div style='font-size:11px;color:#667'>{k}</div><div style='font-size:22px;font-weight:700;color:#1f4e79'>{v}</div></div>"
                    for k, v in kpis(d).items())
    return HTML(f"<div style='display:flex;flex-wrap:wrap'>{cards}</div>")

display(kpi_html(df))

## 3. Segment tables (conversion = subscribed ÷ contacts)

In [ ]:
def rate_table(d, col):
    t = (d.groupby(col, observed=True)
           .agg(contacts=('y_bin','size'), subscribed=('y_bin','sum')).reset_index())
    t['conv_rate_%'] = 100 * t['subscribed'] / t['contacts']
    return t

for c in ['job','month','poutcome','contact','age_group','contact_band']:
    print(f'\n--- {c} ---')
    display(rate_table(df, c).sort_values('conv_rate_%', ascending=False).round(2))

## 4. Dashboard builder (7 linked panels in one figure)

In [ ]:
BLUE, ORANGE = '#1f4e79', '#e8833a'

def hbar(t, col, title_text=None):
    t = t.sort_values('conv_rate_%')
    return go.Bar(x=t['conv_rate_%'], y=t[col].astype(str), orientation='h', marker_color=BLUE,
                  text=[f"{r:.1f}% ({s:,})" for r, s in zip(t['conv_rate_%'], t['subscribed'])],
                  textposition='outside', cliponaxis=False,
                  customdata=t[['subscribed','contacts']], showlegend=False,
                  hovertemplate='%{y}<br>Conversion %{x:.2f}%<br>%{customdata[0]:,} of %{customdata[1]:,} contacts<extra></extra>')

def build_dashboard(d):
    fig = make_subplots(
        rows=3, cols=3, vertical_spacing=0.12, horizontal_spacing=0.10,
        specs=[[{'colspan':3,'secondary_y':True}, None, None], [{},{},{}], [{},{},{}]],
        subplot_titles=['Monthly contact volume vs conversion rate','Previous outcome scorecard',
                        'Conversion rate by job','Conversion by contact frequency',
                        'Conversion by age group','Marital status x education (conv. %)','Subscriptions by channel'])
    # 1. monthly trend (dual axis)
    m = rate_table(d, 'month').sort_values('month')
    fig.add_trace(go.Scatter(x=m['month'].astype(str), y=m['contacts'], name='Contacts', mode='lines+markers',
                             line=dict(color=BLUE)), row=1, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=m['month'].astype(str), y=m['conv_rate_%'], name='Conversion %', mode='lines+markers',
                             line=dict(color=ORANGE)), row=1, col=1, secondary_y=True)
    # 2-5 ranked bars
    fig.add_trace(hbar(rate_table(d,'poutcome'),'poutcome'), row=2, col=1)
    fig.add_trace(hbar(rate_table(d,'job'),'job'), row=2, col=2)
    fig.add_trace(hbar(rate_table(d,'contact_band'),'contact_band'), row=2, col=3)
    fig.add_trace(hbar(rate_table(d,'age_group'),'age_group'), row=3, col=1)
    # 6. heat map
    hm = d.pivot_table(index='education', columns='marital', values='y_bin', aggfunc='mean') * 100
    fig.add_trace(go.Heatmap(z=hm.values, x=hm.columns, y=hm.index, colorscale='Blues', showscale=False,
                             text=np.round(hm.values,1), texttemplate='%{text}%',
                             hovertemplate='%{y} / %{x}: %{z:.1f}%<extra></extra>'), row=3, col=2)
    # 7. channel x outcome (subscriptions)
    co = d.groupby(['contact','poutcome'], observed=True)['y_bin'].agg(['sum','size']).reset_index()
    co['label'] = co['contact'] + ' - ' + co['poutcome']; co['rate'] = 100*co['sum']/co['size']
    co = co.sort_values('sum')
    fig.add_trace(go.Bar(x=co['sum'], y=co['label'], orientation='h', marker_color=co['rate'], marker_colorscale='Blues',
                         text=[f"{s:,} ({r:.1f}%)" for s, r in zip(co['sum'], co['rate'])], textposition='outside',
                         cliponaxis=False, showlegend=False,
                         hovertemplate='%{y}<br>%{x:,} subscribed<extra></extra>'), row=3, col=3)
    fig.update_yaxes(title_text='Contacts', secondary_y=False, row=1, col=1)
    fig.update_yaxes(title_text='Conversion %', secondary_y=True, row=1, col=1)
    fig.update_layout(height=1000, template='plotly_white', legend=dict(orientation='h', y=1.06),
                      margin=dict(l=40, r=40, t=80, b=30), title='Executive Bank Marketing Campaign Dashboard')
    return fig

build_dashboard(df).show()

## 5. Interactive dashboard (dropdown filters)
Choose any combination of filters — the KPI cards and all seven charts update together.

In [ ]:
def opts(col): return ['(All)'] + sorted(df[col].astype(str).unique())
w = {k: widgets.Dropdown(options=opts(c), description=k, style={'description_width':'95px'}, layout=widgets.Layout(width='240px'))
     for k, c in {'Contact Method':'contact','Education':'education','Marital Status':'marital',
                  'Month':'month','Prior Outcome':'poutcome'}.items()}
colmap = {'Contact Method':'contact','Education':'education','Marital Status':'marital','Month':'month','Prior Outcome':'poutcome'}

def update(**kw):
    d = df
    for k, v in kw.items():
        if v != '(All)': d = d[d[colmap[k]].astype(str) == v]
    clear_output(wait=True)
    display(kpi_html(d))
    if len(d) == 0:
        print('No records match this filter combination.'); return
    build_dashboard(d).show()

display(widgets.HBox(list(w.values()), layout=widgets.Layout(flex_flow='row wrap')))
display(widgets.interactive_output(update, w))

## 6. Key insights (computed from the data)

In [ ]:
base = df['y_bin'].mean()*100
print(f'Baseline conversion: {base:.2f}%\n')
for col, label in [('poutcome','Previous outcome'),('contact','Channel'),('month','Month'),('contact_band','Contact frequency')]:
    t = rate_table(df, col).sort_values('conv_rate_%', ascending=False)
    print(label, '->', ', '.join(f"{r[col]} {r['conv_rate_%']:.1f}% (n={r['contacts']:,})" for _, r in t.iterrows()))
j = rate_table(df,'job').sort_values('conv_rate_%', ascending=False)
print('\nTop jobs:', ', '.join(f"{r.job} {r['conv_rate_%']:.1f}%" for _, r in j.head(3).iterrows()))
print('Lowest jobs:', ', '.join(f"{r.job} {r['conv_rate_%']:.1f}%" for _, r in j.tail(2).iterrows()))
may = df[df['month']=='may']; print(f"\nMay share of all calls: {100*len(may)/len(df):.1f}% at {100*may['y_bin'].mean():.2f}% conversion")

### Recommendations
1. Fast-track warm leads (`poutcome` = success / failure) before cold leads.
2. Shift call capacity away from May saturation toward Mar, Sep, Oct, Dec.
3. Prioritise cellular outreach and cap attempts at ~3 per client.
4. Tailor term-deposit bundles for students (17-25) and retirees (65+).
5. Pre-score call lists with the Activity-1 Decision Tree.

**Limitations:** no income/balance column; `duration` is known only after the call; `unknown` codes in six categorical columns.

In [ ]:
# Export the full (unfiltered) dashboard as a standalone interactive HTML file
build_dashboard(df).write_html('bank_dashboard_python.html', include_plotlyjs='cdn')
print('Saved bank_dashboard_python.html')